# Module 2.1: Forward-Backward Algorithm Implementation

## Learning Objectives
By completing this notebook, you will:
1. Implement the forward algorithm to compute observation likelihood P(O|λ)
2. Implement the backward algorithm for state posterior probabilities
3. Compute state occupation probabilities (gamma) and transition probabilities (xi)
4. Apply scaling techniques to prevent numerical underflow
5. Use forward-backward for regime detection in financial time series

## Prerequisites
- Markov chain fundamentals (Module 0)
- HMM framework and parameters (Module 1)
- Dynamic programming concepts
- NumPy proficiency

## Estimated Time: 60 minutes

---

## Setup and Imports

We'll implement the forward-backward algorithm from scratch to understand its mechanics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple
from scipy import stats

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. The Evaluation Problem

**Problem:** Given an HMM with parameters λ = (π, A, B) and observations O = (o₁, o₂, ..., oₜ), compute P(O|λ).

### Naive Approach

Sum over all possible state sequences:
$$P(O|\lambda) = \sum_{\text{all } S} P(O, S | \lambda)$$

For K states and T observations, this requires summing K^T terms - exponential complexity!

### Forward Algorithm Solution

Use dynamic programming to compute in O(T × K²) time.

**Key Idea:** Store partial probabilities and reuse them.

## 2. Forward Algorithm

### Forward Variable Definition

$$\alpha_t(i) = P(o_1, o_2, ..., o_t, q_t = s_i | \lambda)$$

This is the probability of:
- Observing the sequence o₁, ..., oₜ **and**
- Being in state i at time t

### Recursion

**Initialization (t=1):**
$$\alpha_1(i) = \pi_i \cdot b_i(o_1)$$

**Induction (t=2, ..., T):**
$$\alpha_t(j) = \left[\sum_{i=1}^K \alpha_{t-1}(i) \cdot a_{ij}\right] \cdot b_j(o_t)$$

**Termination:**
$$P(O|\lambda) = \sum_{i=1}^K \alpha_T(i)$$

In [ ]:
# Purpose: Implement the forward algorithm for discrete HMMs
# Key Concept: Dynamic programming to compute observation likelihood

def forward_algorithm(
    observations: np.ndarray,
    pi: np.ndarray,
    A: np.ndarray,
    B: np.ndarray
) -> Tuple[np.ndarray, float]:
    """
    Forward algorithm for discrete HMM.
    
    Parameters
    ----------
    observations : ndarray (T,)
        Sequence of observation indices (0 to M-1)
    pi : ndarray (K,)
        Initial state distribution
    A : ndarray (K, K)
        Transition matrix where A[i,j] = P(state_t+1=j | state_t=i)
    B : ndarray (K, M)
        Emission matrix where B[i,m] = P(obs=m | state=i)
    
    Returns
    -------
    alpha : ndarray (T, K)
        Forward variables where alpha[t,i] = P(o_1...o_t, q_t=i | λ)
    log_likelihood : float
        Log of P(O|λ)
    """
    T = len(observations)
    K = len(pi)
    
    # Step 1: Initialize alpha matrix
    alpha = np.zeros((T, K))
    
    # Step 2: Initialization (t=0)
    # α₁(i) = πᵢ × bᵢ(o₁)
    alpha[0] = pi * B[:, observations[0]]
    
    # Step 3: Induction (t=1 to T-1)
    for t in range(1, T):
        for j in range(K):
            # αₜ(j) = [Σᵢ αₜ₋₁(i) × aᵢⱼ] × bⱼ(oₜ)
            alpha[t, j] = np.sum(alpha[t-1] * A[:, j]) * B[j, observations[t]]
    
    # Step 4: Termination
    # P(O|λ) = Σᵢ αₜ(i)
    likelihood = np.sum(alpha[-1])
    log_likelihood = np.log(likelihood + 1e-10)  # Add small constant to avoid log(0)
    
    return alpha, log_likelihood

# Example: Simple weather HMM
# States: 0=Sunny, 1=Rainy
# Observations: 0=Walk, 1=Shop, 2=Clean

pi = np.array([0.6, 0.4])  # Initial: more likely sunny

A = np.array([
    [0.7, 0.3],  # Sunny -> Sunny (0.7), Sunny -> Rainy (0.3)
    [0.4, 0.6]   # Rainy -> Sunny (0.4), Rainy -> Rainy (0.6)
])

B = np.array([
    [0.6, 0.3, 0.1],  # Sunny: Walk (0.6), Shop (0.3), Clean (0.1)
    [0.1, 0.4, 0.5]   # Rainy: Walk (0.1), Shop (0.4), Clean (0.5)
])

# Observation sequence: Walk, Shop, Clean, Shop, Walk
observations = np.array([0, 1, 2, 1, 0])

# Run forward algorithm
alpha, log_lik = forward_algorithm(observations, pi, A, B)

print("Forward Algorithm Results:")
print("="*60)
print(f"Observations: {observations}")
print(f"Observation symbols: ['Walk', 'Shop', 'Clean', 'Shop', 'Walk']")
print(f"\nLog-likelihood: {log_lik:.6f}")
print(f"Likelihood: {np.exp(log_lik):.8e}")
print(f"\nForward variables (α):")
print(alpha)

# Visualize forward variables
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(alpha[:, 0], 'o-', label='α(Sunny)', linewidth=2, markersize=8)
ax.plot(alpha[:, 1], 's-', label='α(Rainy)', linewidth=2, markersize=8)
ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Forward Probability', fontsize=12)
ax.set_title('Forward Variables Over Time', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Scaled Forward Algorithm

For long sequences, α values become extremely small and underflow to zero.

**Solution:** Scale α at each time step to sum to 1.

**Scaling Factor:**
$$c_t = \sum_{i=1}^K \alpha_t(i)$$

**Scaled Alpha:**
$$\hat{\alpha}_t(i) = \frac{\alpha_t(i)}{c_t}$$

**Log-Likelihood:**
$$\log P(O|\lambda) = \sum_{t=1}^T \log c_t$$

In [ ]:
# Purpose: Implement scaled forward algorithm to prevent underflow
# Key Concept: Normalize at each step and track scaling factors

def forward_scaled(
    observations: np.ndarray,
    pi: np.ndarray,
    A: np.ndarray,
    B: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Scaled forward algorithm to prevent numerical underflow.
    
    Returns
    -------
    alpha : ndarray (T, K)
        Scaled forward variables (each time step sums to 1)
    scaling : ndarray (T,)
        Scaling factors at each time step
    log_likelihood : float
        Log P(O|λ) computed from scaling factors
    """
    T = len(observations)
    K = len(pi)
    
    alpha = np.zeros((T, K))
    scaling = np.zeros(T)
    
    # Step 1: Initialization
    alpha[0] = pi * B[:, observations[0]]
    scaling[0] = np.sum(alpha[0])
    alpha[0] /= scaling[0]  # Normalize to sum to 1
    
    # Step 2: Induction with scaling
    for t in range(1, T):
        for j in range(K):
            alpha[t, j] = np.sum(alpha[t-1] * A[:, j]) * B[j, observations[t]]
        
        # Step 3: Scale this time step
        scaling[t] = np.sum(alpha[t])
        alpha[t] /= scaling[t]
    
    # Step 4: Compute log-likelihood from scaling factors
    log_likelihood = np.sum(np.log(scaling))
    
    return alpha, scaling, log_likelihood

# Test on same example
alpha_scaled, scaling, log_lik_scaled = forward_scaled(observations, pi, A, B)

print("Scaled Forward Algorithm:")
print("="*60)
print(f"Log-likelihood: {log_lik_scaled:.6f}")
print(f"\nScaling factors:")
print(scaling)
print(f"\nScaled alpha (each row sums to 1):")
print(alpha_scaled)
print(f"\nRow sums: {alpha_scaled.sum(axis=1)}")

## 4. Backward Algorithm

### Backward Variable Definition

$$\beta_t(i) = P(o_{t+1}, o_{t+2}, ..., o_T | q_t = s_i, \lambda)$$

This is the probability of observing the **future** sequence from t+1 to T, given we're in state i at time t.

### Recursion

**Initialization (t=T):**
$$\beta_T(i) = 1 \quad \text{for all } i$$

**Induction (t=T-1, ..., 1) (backward):**
$$\beta_t(i) = \sum_{j=1}^K a_{ij} \cdot b_j(o_{t+1}) \cdot \beta_{t+1}(j)$$

In [ ]:
# Purpose: Implement backward algorithm
# Key Concept: Compute future observation probability given current state

def backward_algorithm(
    observations: np.ndarray,
    A: np.ndarray,
    B: np.ndarray
) -> np.ndarray:
    """
    Backward algorithm for discrete HMM.
    
    Parameters
    ----------
    observations : ndarray (T,)
        Sequence of observation indices
    A : ndarray (K, K)
        Transition matrix
    B : ndarray (K, M)
        Emission matrix
    
    Returns
    -------
    beta : ndarray (T, K)
        Backward variables where beta[t,i] = P(o_t+1...o_T | q_t=i, λ)
    """
    T = len(observations)
    K = A.shape[0]
    
    beta = np.zeros((T, K))
    
    # Step 1: Initialization (t=T)
    # βₜ(i) = 1 for all states
    beta[-1] = 1.0
    
    # Step 2: Induction (backward from T-1 to 0)
    for t in range(T-2, -1, -1):
        for i in range(K):
            # βₜ(i) = Σⱼ aᵢⱼ × bⱼ(oₜ₊₁) × βₜ₊₁(j)
            beta[t, i] = np.sum(
                A[i, :] * B[:, observations[t+1]] * beta[t+1, :]
            )
    
    return beta

# Run backward algorithm
beta = backward_algorithm(observations, A, B)

print("Backward Algorithm Results:")
print("="*60)
print(f"Backward variables (β):")
print(beta)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(beta[:, 0], 'o-', label='β(Sunny)', linewidth=2, markersize=8)
ax.plot(beta[:, 1], 's-', label='β(Rainy)', linewidth=2, markersize=8)
ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Backward Probability', fontsize=12)
ax.set_title('Backward Variables Over Time', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Scaled Backward Algorithm

We must use the **same scaling factors** from the forward pass.

In [ ]:
# Purpose: Implement scaled backward algorithm
# Key Concept: Use forward scaling factors for consistency

def backward_scaled(
    observations: np.ndarray,
    A: np.ndarray,
    B: np.ndarray,
    scaling: np.ndarray
) -> np.ndarray:
    """
    Scaled backward algorithm using forward scaling factors.
    
    Parameters
    ----------
    scaling : ndarray (T,)
        Scaling factors from forward algorithm
    
    Returns
    -------
    beta : ndarray (T, K)
        Scaled backward variables
    """
    T = len(observations)
    K = A.shape[0]
    
    beta = np.zeros((T, K))
    
    # Step 1: Initialization (scale by last scaling factor)
    beta[-1] = 1.0 / scaling[-1]
    
    # Step 2: Induction (backward)
    for t in range(T-2, -1, -1):
        for i in range(K):
            beta[t, i] = np.sum(
                A[i, :] * B[:, observations[t+1]] * beta[t+1, :]
            )
        # Step 3: Scale by forward scaling factor
        beta[t] /= scaling[t]
    
    return beta

# Run scaled backward
beta_scaled = backward_scaled(observations, A, B, scaling)

print("Scaled Backward Variables:")
print(beta_scaled)

## 6. State Posterior Probabilities (Gamma)

### Gamma Definition

$$\gamma_t(i) = P(q_t = s_i | O, \lambda)$$

Probability of being in state i at time t **given all observations**.

### Computation

$$\gamma_t(i) = \frac{\alpha_t(i) \cdot \beta_t(i)}{P(O|\lambda)} = \frac{\alpha_t(i) \cdot \beta_t(i)}{\sum_j \alpha_t(j) \cdot \beta_t(j)}$$

With scaled variables, this simplifies to:
$$\gamma_t(i) = \hat{\alpha}_t(i) \cdot \hat{\beta}_t(i)$$

Then normalize.

In [ ]:
# Purpose: Compute state posterior probabilities
# Key Concept: Combine forward and backward to get full posterior

def compute_gamma(
    alpha: np.ndarray,
    beta: np.ndarray
) -> np.ndarray:
    """
    Compute state posterior probabilities γₜ(i) = P(qₜ=i | O, λ).
    
    Parameters
    ----------
    alpha : ndarray (T, K)
        Scaled forward variables
    beta : ndarray (T, K)
        Scaled backward variables
    
    Returns
    -------
    gamma : ndarray (T, K)
        State posteriors where gamma[t,i] = P(q_t=i | O, λ)
    """
    # Step 1: Multiply alpha and beta element-wise
    gamma = alpha * beta
    
    # Step 2: Normalize each time step to sum to 1
    gamma /= gamma.sum(axis=1, keepdims=True)
    
    return gamma

# Compute gamma
gamma = compute_gamma(alpha_scaled, beta_scaled)

print("State Posterior Probabilities (Gamma):")
print("="*60)
print(gamma)
print(f"\nRow sums (should be 1): {gamma.sum(axis=1)}")

# Visualize state probabilities
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(observations))
ax.bar(x - 0.2, gamma[:, 0], width=0.4, label='P(Sunny | O)', color='gold', edgecolor='black')
ax.bar(x + 0.2, gamma[:, 1], width=0.4, label='P(Rainy | O)', color='steelblue', edgecolor='black')
ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('State Posterior Probabilities at Each Time Step', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(['Walk', 'Shop', 'Clean', 'Shop', 'Walk'])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nMost Likely State at Each Time:")
for t in range(len(observations)):
    state = 'Sunny' if gamma[t, 0] > gamma[t, 1] else 'Rainy'
    print(f"  t={t}: {state} (P={max(gamma[t]):.3f})")

## 7. Transition Posterior Probabilities (Xi)

### Xi Definition

$$\xi_t(i, j) = P(q_t = s_i, q_{t+1} = s_j | O, \lambda)$$

Probability of transitioning from state i to state j at time t.

### Computation

$$\xi_t(i, j) = \frac{\alpha_t(i) \cdot a_{ij} \cdot b_j(o_{t+1}) \cdot \beta_{t+1}(j)}{P(O|\lambda)}$$

In [ ]:
# Purpose: Compute transition posterior probabilities
# Key Concept: Probability of specific transitions given all observations

def compute_xi(
    observations: np.ndarray,
    alpha: np.ndarray,
    beta: np.ndarray,
    A: np.ndarray,
    B: np.ndarray
) -> np.ndarray:
    """
    Compute transition posterior probabilities ξₜ(i,j) = P(qₜ=i, qₜ₊₁=j | O, λ).
    
    Returns
    -------
    xi : ndarray (T-1, K, K)
        Transition posteriors where xi[t,i,j] = P(q_t=i, q_t+1=j | O, λ)
    """
    T = len(observations)
    K = A.shape[0]
    
    xi = np.zeros((T-1, K, K))
    
    # Step 1: For each time step
    for t in range(T-1):
        denominator = 0.0
        
        # Step 2: Compute numerator for all (i,j) pairs
        for i in range(K):
            for j in range(K):
                xi[t, i, j] = (
                    alpha[t, i] *
                    A[i, j] *
                    B[j, observations[t+1]] *
                    beta[t+1, j]
                )
                denominator += xi[t, i, j]
        
        # Step 3: Normalize
        xi[t] /= (denominator + 1e-10)
    
    return xi

# Compute xi
xi = compute_xi(observations, alpha_scaled, beta_scaled, A, B)

print("Transition Posterior Probabilities (Xi):")
print("="*60)
print(f"Shape: {xi.shape} (T-1, K, K)")
print(f"\nXi at t=0 (after observing 'Walk'):")
print(xi[0])
print(f"\nVerification - sum over all transitions at t=0: {xi[0].sum():.4f}")

## Exercise 1: Complete Forward-Backward Implementation

**Task:** Implement a complete forward-backward function that returns both gamma and xi.

**Expected Output:** Function that takes HMM parameters and returns state and transition posteriors.

In [ ]:
# YOUR CODE HERE
# ---------------

def forward_backward(
    observations: np.ndarray,
    pi: np.ndarray,
    A: np.ndarray,
    B: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Complete forward-backward algorithm.
    
    Returns
    -------
    gamma : ndarray (T, K)
        State posteriors
    xi : ndarray (T-1, K, K)
        Transition posteriors
    log_likelihood : float
        Log P(O|λ)
    """
    # YOUR IMPLEMENTATION HERE
    # Use the functions you've seen above
    
    return None, None, None

# Test your implementation
gamma_test, xi_test, log_lik_test = forward_backward(observations, pi, A, B)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_1():
    assert gamma_test is not None, "Don't forget to implement forward_backward!"
    assert xi_test is not None, "Xi should not be None"
    assert log_lik_test is not None, "Log-likelihood should not be None"
    
    # Check shapes
    T, K = len(observations), 2
    assert gamma_test.shape == (T, K), f"Gamma shape should be ({T}, {K})"
    assert xi_test.shape == (T-1, K, K), f"Xi shape should be ({T-1}, {K}, {K})"
    
    # Check normalization
    assert np.allclose(gamma_test.sum(axis=1), 1.0), "Gamma rows should sum to 1"
    assert np.allclose(xi_test.sum(axis=(1,2)), 1.0), "Xi should sum to 1 at each time"
    
    # Check against reference implementation
    assert np.allclose(gamma_test, gamma, atol=1e-6), "Gamma doesn't match reference"
    
    print("✅ Exercise 1 passed!")
    print(f"\nLog-likelihood: {log_lik_test:.6f}")

test_exercise_1()

<details>
<summary>Hint 1</summary>
Call forward_scaled first to get alpha and scaling factors, then backward_scaled with those scaling factors.
</details>

<details>
<summary>Hint 2</summary>
Use compute_gamma and compute_xi functions with the scaled alpha and beta.
</details>

## Exercise 2: Market Regime Detection

**Task:** Apply forward-backward to detect Bull/Bear regimes in simulated market data.

Generate returns from a 2-state Gaussian HMM, then use forward-backward to recover the hidden states.

**Expected Output:** Plot comparing true states vs. decoded probabilities.

In [ ]:
# Generate synthetic market data
np.random.seed(123)
T = 200

# True HMM parameters
pi_market = np.array([0.6, 0.4])
A_market = np.array([[0.95, 0.05], [0.10, 0.90]])
means = np.array([0.0008, -0.0015])  # Bull: +0.08% daily, Bear: -0.15% daily
stds = np.array([0.01, 0.025])  # Bull: 1% vol, Bear: 2.5% vol

# Simulate true states
true_states = [np.random.choice(2, p=pi_market)]
for _ in range(T-1):
    true_states.append(np.random.choice(2, p=A_market[true_states[-1]]))
true_states = np.array(true_states)

# Generate returns
returns = np.array([
    np.random.normal(means[s], stds[s]) for s in true_states
])

# YOUR CODE HERE
# ---------------

# Step 1: For Gaussian HMM, you need to compute emission probabilities
# B is not a discrete matrix - compute Gaussian densities

def gaussian_emission_probs(returns, means, stds):
    """
    Compute emission probability matrix for Gaussian HMM.
    
    Returns B where B[t, k] = P(return_t | state=k)
    """
    # YOUR IMPLEMENTATION HERE
    return None

# Step 2: Adapt forward-backward for continuous emissions
# Step 3: Decode states using gamma
# Step 4: Compare with true states

gamma_market = None  # Replace with your implementation
decoded_states = None  # Replace with your implementation

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2():
    assert gamma_market is not None, "Don't forget to compute gamma_market!"
    assert decoded_states is not None, "Don't forget to decode states!"
    
    # Check shapes
    assert gamma_market.shape == (T, 2), f"Gamma shape should be ({T}, 2)"
    assert len(decoded_states) == T, f"Decoded states length should be {T}"
    
    # Check accuracy
    accuracy = np.mean(decoded_states == true_states)
    assert accuracy > 0.7, f"Decoding accuracy {accuracy:.2%} is too low (should be > 70%)"
    
    print("✅ Exercise 2 passed!")
    print(f"\nDecoding accuracy: {accuracy:.1%}")
    
    # Visualize
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Returns
    axes[0].plot(returns, 'k-', alpha=0.7)
    axes[0].set_ylabel('Return')
    axes[0].set_title('Simulated Market Returns')
    axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
    
    # True states
    axes[1].fill_between(range(T), true_states, alpha=0.6, step='mid')
    axes[1].set_ylabel('State')
    axes[1].set_title('True Hidden States')
    axes[1].set_yticks([0, 1])
    axes[1].set_yticklabels(['Bull', 'Bear'])
    
    # Decoded probabilities
    axes[2].plot(gamma_market[:, 0], label='P(Bull | returns)', linewidth=2)
    axes[2].plot(gamma_market[:, 1], label='P(Bear | returns)', linewidth=2)
    axes[2].set_xlabel('Time')
    axes[2].set_ylabel('Probability')
    axes[2].set_title('Decoded State Probabilities (Forward-Backward)')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()

test_exercise_2()

<details>
<summary>Hint 1</summary>
Use scipy.stats.norm.pdf to compute Gaussian densities: pdf(return, loc=mean, scale=std).
</details>

<details>
<summary>Hint 2</summary>
For continuous emissions, replace the discrete B matrix lookup B[i, obs[t]] with the continuous density B_continuous[t, i].
</details>

## Summary

### Key Takeaways

1. **Forward algorithm** computes P(O|λ) efficiently in O(T×K²) time using dynamic programming
2. **Backward algorithm** computes future observation probabilities given current state
3. **Scaling** is essential to prevent numerical underflow in long sequences
4. **Gamma (γ)** gives state posterior probabilities using all observations
5. **Xi (ξ)** gives transition posterior probabilities, used in learning (Baum-Welch)

### Relationship Summary

```
Forward (α):  Past → Present     P(o₁...oₜ, qₜ=i)
Backward (β): Future ← Present   P(oₜ₊₁...oₜ | qₜ=i)
Combined:     Full posterior     P(qₜ=i | O) ∝ α(i) × β(i)
```

### What's Next

In the next notebook, we'll implement the **Viterbi algorithm** to find the most likely state sequence (not just marginal probabilities).

---

**Next Notebook:** `02_viterbi_impl.ipynb` - Finding optimal state sequences